In [ ]:
# Sequence ontology file will be used for checking if a feature.type is transcripted or not
# Downloaded from https://github.com/The-Sequence-Ontology/SO-Ontologies

In [38]:
from scripts.canonicalize_genomes import parse_ncbi_assembly
json_path = "../data/hg38/ncbi_dataset/data/GCF_000001405.26/sequence_report.jsonl"
parsed = parse_ncbi_assembly(json_path)
chromosome_names = []
for l in parsed:
    if l["role"] == "assembled-molecule":
        chromosome_names.append(l["refseqAccession"])
        print(l)

{'assemblyAccession': 'GCF_000001405.26', 'assemblyUnit': 'Primary Assembly', 'assignedMoleculeLocationType': 'Chromosome', 'chrName': '1', 'gcCount': '103674491', 'gcPercent': 41.5, 'genbankAccession': 'CM000663.2', 'length': 248956422, 'refseqAccession': 'NC_000001.11', 'role': 'assembled-molecule', 'sequenceName': '1', 'ucscStyleName': 'chr1', 'unlocalizedCount': 9}
{'assemblyAccession': 'GCF_000001405.26', 'assemblyUnit': 'Primary Assembly', 'assignedMoleculeLocationType': 'Chromosome', 'chrName': '2', 'gcCount': '101284083', 'gcPercent': 40.5, 'genbankAccession': 'CM000664.2', 'length': 242193529, 'refseqAccession': 'NC_000002.12', 'role': 'assembled-molecule', 'sequenceName': '2', 'ucscStyleName': 'chr2', 'unlocalizedCount': 2}
{'assemblyAccession': 'GCF_000001405.26', 'assemblyUnit': 'Primary Assembly', 'assignedMoleculeLocationType': 'Chromosome', 'chrName': '3', 'gcCount': '91922884', 'gcPercent': 40.0, 'genbankAccession': 'CM000665.2', 'length': 198295559, 'refseqAccession': 

In [37]:
print(len(chromosome_names))

25


In [52]:
from BCBio import GFF

gff_path = "../data/hg38/ncbi_dataset/data/GCF_000001405.26/genomic.gff"

limit_info = {
    "gff_id": chromosome_names
    #"gff_id": ["NC_000001.11"]
}

records = []

with open(gff_path, "r") as f:
    for rec in GFF.parse(f):
        records.append(rec)

In [80]:
import pronto

so = pronto.Ontology("../so.clean.obo")
name_to_term = { # maps name (like "transcript") to Term object
    term.name: term
    for term in so.terms()
    if term.name is not None and not term.obsolete
}

def is_transcribed(name):
    global name_to_term
    ancestors = ("transcript","mRNA", "primary_transcript")

    term = name_to_term.get(name)
    if term is None:
        print(f"Warning: {name} not found in Sequence Ontology")
        return False
    superclases = term.superclasses(with_self=True)

    for a in ancestors:
        a_term = name_to_term.get(name)
        if a_term in superclases:
            return True

    return False

print(is_transcribed("tRNA"))

True


In [ ]:
# skip_deeper_exploration = (
#     "primary_transcript", # Generic category for RNA
#     "mRNA", # Mature RNA 
#     "antisense_RNA", # RNA that's transcribed in the - direction relative to the arbitrary genome orientation
#     "rRNA", # Ribosomal RNA
#     "miRNA", # Micro RNA
#     "snoRNA", # Small nucleolar RNA, guides stuff in mRNA
#     "snRNA", # Small nuclear RNA
#     "tRNA", # Transfer RNA
#     "telomerase_RNA", # The RNA component of the telomerase
#     "RNase_MRP_RNA", # RNA of RNase MRP
#     "RNase_P_RNA", # RNA of ribonuclease P
#     "lnc_RNA", # Long non-coding RNA
# )

feature_types = set()

def explore_feature(feature):
    global feature_types

    if feature.type in skip_deeper_exploration:
        return

    feature_types.add(feature.type)
    for sub_f in feature.sub_features:
        explore_feature(sub_f)

for rec in records:
    for feature in rec.features:
        explore_feature(feature)

In [60]:
print(feature_types)

{'D_gene_segment', 'CDS', 'mRNA', 'D_loop', 'primary_transcript', 'C_gene_segment', 'J_gene_segment', 'match', 'snoRNA', 'exon', 'cDNA_match', 'pseudogene', 'gene', 'telomerase_RNA', 'lnc_RNA', 'biological_region', 'RNase_MRP_RNA', 'centromere', 'tRNA', 'miRNA', 'V_gene_segment', 'antisense_RNA', 'snRNA', 'region', 'rRNA', 'sequence_feature', 'RNase_P_RNA'}
